# PDF → TXT (PyMuPDF)

Este trecho lê um arquivo PDF e salva **todo o texto** em um `.txt` usando **PyMuPDF** (`fitz`).

**Pré-requisito**: `pip install pymupdf` (se ainda não estiver instalado).

- Ajuste `pdf_path` para apontar para o PDF desejado.
- O arquivo `.txt` será salvo em `out_txt_path`.

In [ ]:

# 1) Imports (PyMuPDF é importado como `fitz`)
from pathlib import Path

import fitz  # pip install pymupdf

# 2) Função genérica: PDF -> TXT (extração simples por página)
def pdf_to_txt(pdf_path: str | Path, out_txt_path: str | Path | None = None) -> Path:
    """Extrai todo o texto de um PDF e salva em um arquivo .txt (UTF-8).

    Observação: para PDFs com texto em múltiplas colunas,
    use a função 'pdf_to_txt_layout_aware' na célula seguinte.

    Args:
        pdf_path: caminho do PDF de entrada.
        out_txt_path: caminho do .txt de saída (opcional). Se None, usa o mesmo nome do PDF com .txt.

    Returns:
        Path do arquivo .txt gerado.
    """
    pdf_path = Path(pdf_path)
    if out_txt_path is None:
        out_txt_path = pdf_path.with_suffix(".txt")
    out_txt_path = Path(out_txt_path)

    # Abre o PDF e concatena o texto de cada página, na ordem
    text_parts: list[str] = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text_parts.append(page.get_text("text"))

    # Escreve o resultado final em UTF-8
    out_txt_path.parent.mkdir(parents=True, exist_ok=True)
    out_txt_path.write_text("\n".join(text_parts), encoding="utf-8")
    return out_txt_path

# 3) Exemplo de uso: UNIAO (Manifesto)

pdf_path = Path("../data/political agenda/PSOL/PAUTA-PARTIDARIA-PSOL.pdf")
out_txt_path = Path("../data/political agenda/PSOL/txt/PAUTA-PARTIDARIA-PSOL.txt")

saved_path = pdf_to_txt(pdf_path, out_txt_path)
print(f"TXT salvo em: {saved_path}")

TXT salvo em: ../data/political agenda/MDB/txt/MDB-2023-2027-1.txt


In [8]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable

import fitz  # pip install pymupdf

def _clean_extracted_text(text: str) -> str:
    # Remove espaços no fim de linha, normaliza quebras e junta hifenização comum de PDF
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Junta palavras quebradas por hífen no fim da linha: "gover-\no" -> "governo"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text.strip()

def _guess_column_split_x(block_x0s: list[float], page_width: float) -> float | None:
    """Tenta detectar 2 colunas achando o maior 'gap' em x0.

    Retorna o x de corte (split) ou None (tratando como 1 coluna).
    """
    if len(block_x0s) < 6:
        return None
    xs = sorted(block_x0s)
    gaps = [(xs[i + 1] - xs[i], i) for i in range(len(xs) - 1)]
    max_gap, idx = max(gaps, key=lambda t: t[0])
    # Heurística: gap grande o suficiente sugere 2 colunas
    if max_gap < page_width * 0.15:
        return None
    return (xs[idx] + xs[idx + 1]) / 2

def _iter_page_text_blocks(page: fitz.Page) -> Iterable[tuple[float, float, float, float, str]]:
    """Itera blocos de texto (x0,y0,x1,y1,text) ignorando imagens/retângulos."""
    for b in page.get_text("blocks"):
        x0, y0, x1, y1, text, _block_no, block_type = b
        if block_type != 0:
            continue
        text = (text or "").strip()
        if not text:
            continue
        yield float(x0), float(y0), float(x1), float(y1), text

def extract_page_text_layout_aware(page: fitz.Page) -> str:
    """Extrai texto respeitando melhor a ordem de leitura em páginas com colunas.

    Estratégia:
    - pega blocos com coordenadas (PyMuPDF 'blocks')
    - detecta 1 vs 2 colunas via gap em x0
    - ordena por coluna (esquerda -> direita) e, dentro, por y0
    """
    rect = page.rect
    page_width = float(rect.width)
    blocks = list(_iter_page_text_blocks(page))
    if not blocks:
        return ""

    split_x = _guess_column_split_x([b[0] for b in blocks], page_width)
    if split_x is None:
        # 1 coluna: ordena por y (e depois x)
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))
    else:
        left = [b for b in blocks if b[0] < split_x]
        right = [b for b in blocks if b[0] >= split_x]
        left_sorted = sorted(left, key=lambda b: (b[1], b[0]))
        right_sorted = sorted(right, key=lambda b: (b[1], b[0]))
        blocks_sorted = left_sorted + right_sorted

    text = "\n".join(b[4] for b in blocks_sorted)
    return _clean_extracted_text(text)

def pdf_to_txt_layout_aware(pdf_path: str | Path, out_txt_path: str | Path | None = None) -> Path:
    """PDF -> TXT com extração por blocos e reordenação para colunas."""
    pdf_path = Path(pdf_path)
    if out_txt_path is None:
        out_txt_path = pdf_path.with_suffix(".txt")
    out_txt_path = Path(out_txt_path)

    page_texts: list[str] = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            page_texts.append(extract_page_text_layout_aware(page))

    out_txt_path.parent.mkdir(parents=True, exist_ok=True)
    out_txt_path.write_text("\n\n".join(t for t in page_texts if t), encoding="utf-8")
    return out_txt_path

# /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/
# Exemplo usando o manifesto da UNIAO (salva em pasta txt/)
# data/political agenda/NOVO/APRESENTACAO_PARTIDO_NOVO-2025-v2-1.pdf
pdf_path = Path("../data/political agenda/NOVO/APRESENTACAO_PARTIDO_NOVO-2025-v2-1.pdf")
out_txt_path = Path("../data/political agenda/NOVO/txt/APRESENTACAO_PARTIDO_NOVO-2025-v2-1.txt")

saved_path = pdf_to_txt_layout_aware(pdf_path, out_txt_path)
print(f"TXT (layout-aware) salvo em: {saved_path}")

TXT (layout-aware) salvo em: ../data/political agenda/NOVO/txt/APRESENTACAO_PARTIDO_NOVO-2025-v2-1.txt


In [ ]:
# PDF -> TXT (PL) com 2 colunas, tabelas e remoção de cabeçalho/rodapé/notas
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import fitz  # PyMuPDF


@dataclass(frozen=True)
class _Frag:
    x0: float
    y0: float
    x1: float
    y1: float
    text: str
    size: float


def _norm_line(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    return s


def _clean_text_basic(text: str) -> str:
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text.strip()


def _iter_frags_dict(page: fitz.Page) -> Iterable[_Frag]:
    """Itera fragmentos de texto com bbox e tamanho de fonte (bom p/ tabelas)."""
    d = page.get_text("dict")
    for block in d.get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                txt = span.get("text", "")
                if not txt or not txt.strip():
                    continue
                x0, y0, x1, y1 = span.get("bbox", (0, 0, 0, 0))
                size = float(span.get("size", 0.0) or 0.0)
                yield _Frag(float(x0), float(y0), float(x1), float(y1), txt, size)


def _group_frags_into_lines(frags: list[_Frag], y_tol: float) -> list[list[_Frag]]:
    """Agrupa fragmentos por linha (aprox) usando tolerância em y0."""
    frags_sorted = sorted(frags, key=lambda f: (f.y0, f.x0))
    lines: list[list[_Frag]] = []
    for frag in frags_sorted:
        if not lines:
            lines.append([frag])
            continue
        last_line = lines[-1]
        last_y = sum(f.y0 for f in last_line) / len(last_line)
        if abs(frag.y0 - last_y) <= y_tol:
            last_line.append(frag)
        else:
            lines.append([frag])
    # Ordena cada linha por x0
    for ln in lines:
        ln.sort(key=lambda f: f.x0)
    return lines


def _guess_two_columns_split_x(xs: list[float], page_width: float) -> Optional[float]:
    if len(xs) < 12:
        return None
    xs = sorted(xs)
    gaps = [(xs[i + 1] - xs[i], i) for i in range(len(xs) - 1)]
    max_gap, idx = max(gaps, key=lambda t: t[0])
    if max_gap < page_width * 0.18:
        return None
    return (xs[idx] + xs[idx + 1]) / 2


def _render_lines_as_text(lines: list[list[_Frag]], *, join_tables_with_tabs: bool = True) -> str:
    out_lines: list[str] = []
    for ln in lines:
        parts = [_norm_line(f.text) for f in ln]
        parts = [p for p in parts if p]
        if not parts:
            continue
        # Heurística simples: se há muitos "pedaços" na mesma linha, tende a ser tabela
        if join_tables_with_tabs and len(parts) >= 3:
            out_lines.append("\t".join(parts))
        else:
            out_lines.append(" ".join(parts))
    return "\n".join(out_lines)


def extract_page_text_pl(page: fitz.Page) -> str:
    """Extrai texto tentando respeitar colunas, preservando linhas (bom p/ tabelas).

    Remove:
      - Cabeçalho: topo ~8%
      - Rodapé: base ~10%
      - Notas de rodapé: fonte pequena + parte inferior
    """
    rect = page.rect
    page_w, page_h = float(rect.width), float(rect.height)
    header_y = page_h * 0.08
    footer_y = page_h * 0.90

    frags_all = list(_iter_frags_dict(page))
    if not frags_all:
        return ""

    sizes = [f.size for f in frags_all if f.size > 0]
    median_size = sorted(sizes)[len(sizes) // 2] if sizes else 0.0
    footnote_max_size = median_size * 0.88 if median_size else 0.0

    # 1) Filtra áreas fixas (cabeçalho/rodapé) e notas (fonte pequena no fim)
    frags: list[_Frag] = []
    for f in frags_all:
        if f.y1 <= header_y:
            continue
        if f.y0 >= footer_y:
            continue
        if footnote_max_size and f.size <= footnote_max_size and f.y0 >= page_h * 0.75:
            continue
        frags.append(f)

    if not frags:
        return ""

    # 2) Decide 1 vs 2 colunas
    split_x = _guess_two_columns_split_x([f.x0 for f in frags], page_w)

    # 3) Agrupa em linhas com tolerância proporcional ao tamanho de fonte
    y_tol = max(2.0, (median_size * 0.55) if median_size else 3.0)

    if split_x is None:
        lines = _group_frags_into_lines(frags, y_tol=y_tol)
        page_text = _render_lines_as_text(lines)
    else:
        left_frags = [f for f in frags if f.x0 < split_x]
        right_frags = [f for f in frags if f.x0 >= split_x]
        left_lines = _group_frags_into_lines(left_frags, y_tol=y_tol)
        right_lines = _group_frags_into_lines(right_frags, y_tol=y_tol)
        page_text = _render_lines_as_text(left_lines) + "\n\n" + _render_lines_as_text(right_lines)

    return _clean_text_basic(page_text)


def _remove_repeated_header_like_lines(pages: list[str], *, min_ratio: float = 0.35) -> list[str]:
    """Remove linhas curtas que se repetem em muitas páginas (ex: cabeçalho residual)."""
    all_lines = []
    for txt in pages:
        # pega só primeiras/últimas linhas (onde sobram resquícios)
        lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
        all_lines.extend(lines[:3] + lines[-3:])

    if not all_lines or len(pages) < 3:
        return pages

    counts = Counter(_norm_line(ln) for ln in all_lines if 3 <= len(_norm_line(ln)) <= 80)
    threshold = max(2, int(len(pages) * min_ratio))
    repeated = {ln for ln, c in counts.items() if c >= threshold}
    if not repeated:
        return pages

    cleaned_pages: list[str] = []
    for txt in pages:
        kept = []
        for ln in txt.splitlines():
            n = _norm_line(ln)
            if n in repeated:
                continue
            kept.append(ln)
        cleaned_pages.append("\n".join(kept).strip())
    return cleaned_pages


def pdf_to_txt_pl(pdf_path: str | Path, out_txt_path: str | Path | None = None) -> Path:
    pdf_path = Path(pdf_path)
    if out_txt_path is None:
        out_txt_path = pdf_path.with_suffix(".txt")
    out_txt_path = Path(out_txt_path)

    page_texts: list[str] = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            page_texts.append(extract_page_text_pl(page))

    page_texts = _remove_repeated_header_like_lines(page_texts)
    final_text = "\n\n".join(t for t in page_texts if t).strip()
    out_txt_path.parent.mkdir(parents=True, exist_ok=True)
    out_txt_path.write_text(final_text, encoding="utf-8")
    return out_txt_path


# --- Execução para o PDF do PL ---
pdf_path = Path("../data/political agenda/MDB/conteudo programatico mdb.pdf")
out_txt_path = Path("../data/political agenda/MDB/txt/conteudo programatico mdb.txt")

saved_path = pdf_to_txt_pl(pdf_path, out_txt_path)
print(f"TXT salvo em: {saved_path}")

TXT salvo em: ../data/political agenda/PL/txt/conteudo_programatico_mdb.txt


In [13]:
# OCR: PDF (imagem/scaneado) -> TXT (MDB-2023-2027-1.pdf)
# Requisitos:
#   - pip install pytesseract pillow
#   - binário do sistema: tesseract-ocr (e idioma 'por' se quiser PT-BR)

from pathlib import Path
import re
import shutil
from typing import Iterable, Optional

import fitz  # PyMuPDF

try:
    import pytesseract
    from PIL import Image, ImageOps
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Dependências faltando. Rode: pip install pytesseract pillow"
    ) from e


def _clean_ocr_text(text: str) -> str:
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Junta hifenização comum de OCR/PDF
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # Remove múltiplos espaços
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def _page_pixmap(
    page: fitz.Page,
    *,
    dpi: int = 300,
    header_ratio: float = 0.08,
    footer_ratio: float = 0.92,
) -> fitz.Pixmap:
    """Renderiza a página como imagem (com crop para remover cabeçalho/rodapé)."""
    rect = page.rect
    page_w, page_h = float(rect.width), float(rect.height)
    clip = fitz.Rect(0, page_h * header_ratio, page_w, page_h * footer_ratio)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    return page.get_pixmap(matrix=mat, clip=clip, alpha=False)


def _pixmap_to_pil(pix: fitz.Pixmap) -> Image.Image:
    mode = "RGB" if pix.n < 4 else "RGBA"
    img = Image.frombytes(mode, (pix.width, pix.height), pix.samples)
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img


def _preprocess_for_ocr(img: Image.Image) -> Image.Image:
    # Grayscale + autocontrast (melhora OCR em scans)
    img = ImageOps.grayscale(img)
    img = ImageOps.autocontrast(img)
    return img


def pdf_ocr_to_txt(
    pdf_path: str | Path,
    out_txt_path: str | Path | None = None,
    *,
    lang: str = "por",
    dpi: int = 300,
    header_ratio: float = 0.08,
    footer_ratio: float = 0.92,
    max_pages: int | None = None,
) -> Path:
    pdf_path = Path(pdf_path)
    if out_txt_path is None:
        out_txt_path = pdf_path.with_suffix(".txt")
    out_txt_path = Path(out_txt_path)

    if shutil.which("tesseract") is None:
        raise RuntimeError(
            "Tesseract não encontrado no sistema. Instale (Ubuntu/Debian): "
            "sudo apt-get install tesseract-ocr tesseract-ocr-por"
        )

    page_texts: list[str] = []
    with fitz.open(pdf_path) as doc:
        total = len(doc)
        limit = min(total, max_pages) if max_pages is not None else total
        for i in range(limit):
            page = doc.load_page(i)
            pix = _page_pixmap(
                page, dpi=dpi, header_ratio=header_ratio, footer_ratio=footer_ratio
            )
            img = _preprocess_for_ocr(_pixmap_to_pil(pix))

            config = "--psm 6"  # bloco de texto uniforme; costuma funcionar bem p/ páginas com parágrafos
            try:
                text = pytesseract.image_to_string(img, lang=lang, config=config)
            except pytesseract.TesseractError:
                # fallback se o idioma 'por' não estiver instalado
                text = pytesseract.image_to_string(img, lang="eng", config=config)

            text = _clean_ocr_text(text)
            if text:
                page_texts.append(text)
            print(f"OCR página {i+1}/{limit} -> {len(text)} chars")

    final_text = "\n\n".join(page_texts).strip()
    out_txt_path.parent.mkdir(parents=True, exist_ok=True)
    out_txt_path.write_text(final_text, encoding="utf-8")
    return out_txt_path


# --- Execução (MDB) ---
pdf_path = Path("../data/political agenda/MDB/MDB-2023-2027-1.pdf")
out_txt_path = Path("../data/political agenda/MDB/txt/MDB-2023-2027-1.ocr.txt")

# Dica: se quiser testar rápido, use max_pages=1
saved_path = pdf_ocr_to_txt(pdf_path, out_txt_path, lang="por", max_pages=None)
print(f"TXT (OCR) salvo em: {saved_path}")

OCR página 1/1 -> 2011 chars
TXT (OCR) salvo em: ../data/political agenda/MDB/txt/MDB-2023-2027-1.ocr.txt
